# Spark Setup

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITR7_Time_Series")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "8")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 23:33:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Data Cleaning

In [2]:
from pyspark.sql import functions as F

# Load data
commentsDF = spark.read.json("/home/amritap1/30123/final/comments")
submissionsDF = spark.read.json("/home/amritap1/30123/final/submissions")

25/11/20 23:34:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [6]:
print(f"Comments: {commentsDF.count()}")
print(f"Submissions: {submissionsDF.count()}")

Comments: 3396713
Submissions: 190347


In [27]:
# Clean and prepare data with timestamps
def clean_time_series_data(df, text_column):
    df = df.filter(F.col(text_column).isNotNull())
    df = df.filter(~F.col(text_column).isin("[deleted]", "[removed]"))
    
    # Remove bots
    a = F.lower(F.col("author"))
    is_bot = F.coalesce(
        a.contains("automoderator") | a.contains("moderator") | a.contains("bot") | a.contains("automod"),
        F.lit(False)
    )
    df = df.filter(~is_bot)
    
    # Convert timestamps
    df = df.withColumn(
        "created_ts",
        F.coalesce(
            F.from_unixtime(F.col("created").cast("double")).cast("timestamp"),
            F.to_timestamp(F.col("created_utc"))
        )
    )
    
    return df

# Apply cleaning
commentsDF_clean = clean_time_series_data(commentsDF, "body")
submissionsDF_clean = clean_time_series_data(submissionsDF, "selftext")

In [28]:
# Filter out records with NULL timestamps
commentsDF_clean = commentsDF_clean.filter(F.col("created_ts").isNotNull())
submissionsDF_clean = submissionsDF_clean.filter(F.col("created_ts").isNotNull())

print(f"Comments after filtering: {commentsDF_clean.count()}")
print(f"Submissions after filtering: {submissionsDF_clean.count()}")

Comments after filtering: 72
Submissions after filtering: 18923


In [29]:
print(f"Submissions after filtering: {submissionsDF_clean.filter(F.col("num_comments").isNotNull()).count()}")

[Stage 137:======================================>                  (4 + 2) / 6]

Submissions after filtering: 18923


## Time Series Aggregation

In [30]:
# Aggregate by day
daily_activity = (submissionsDF_clean
    .withColumn("date", F.to_date("created_ts"))
    .groupBy("date")
    .agg(
        F.count("*").alias("submission_count"),
        F.sum("num_comments").alias("total_comments"),
        F.avg("num_comments").alias("avg_comments_per_post"),
        F.avg("score").alias("avg_score")
    )
    .orderBy("date")
)

daily_activity.show(20)

[Stage 140:===============================================>         (5 + 1) / 6]

+----------+----------------+--------------+---------------------+------------------+
|      date|submission_count|total_comments|avg_comments_per_post|         avg_score|
+----------+----------------+--------------+---------------------+------------------+
|2013-12-31|               4|            89|                22.25|              7.75|
|2014-01-01|               6|            56|    9.333333333333334|3.6666666666666665|
|2014-01-02|              10|           134|                 13.4|               8.8|
|2014-01-03|               3|            71|   23.666666666666668| 8.333333333333334|
|2014-01-04|               1|             3|                  3.0|               0.0|
|2014-01-05|               3|             8|   2.6666666666666665|               2.0|
|2014-01-06|               8|            37|                4.625|               2.5|
|2014-01-07|               4|            69|                17.25|              1.75|
|2014-01-08|               6|            33|          

In [31]:
# Weekly aggregation
weekly_activity = (daily_activity
    .withColumn("week", F.weekofyear("date"))
    .withColumn("year", F.year("date"))
    .groupBy("year", "week")
    .agg(
        F.sum("submission_count").alias("weekly_submissions"),
        F.sum("total_comments").alias("weekly_comments"),
        F.avg("avg_score").alias("avg_weekly_score")
    )
    .orderBy("year", "week")
)

# Monthly aggregation
monthly_activity = (daily_activity
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .groupBy("year", "month")
    .agg(
        F.sum("submission_count").alias("monthly_submissions"),
        F.sum("total_comments").alias("monthly_comments"),
        F.avg("avg_score").alias("avg_monthly_score")
    )
    .orderBy("year", "month")
)

monthly_activity.show(20)

[Stage 143:======================================>                  (4 + 2) / 6]

+----+-----+-------------------+----------------+------------------+
|year|month|monthly_submissions|monthly_comments| avg_monthly_score|
+----+-----+-------------------+----------------+------------------+
|2013|   12|                  4|              89|              7.75|
|2014|    1|                225|            3204| 6.002369756680102|
|2014|    2|                297|            3161|5.8154668808589625|
|2014|    3|                268|            2340| 6.274830556329609|
|2014|    4|                152|            1653| 7.340762118539897|
|2014|    5|                164|            1452| 5.799497354497353|
|2014|    6|                136|            1101| 5.278927203065134|
|2014|    7|                156|            1341| 4.790874304267162|
|2014|    8|                101|             820| 4.707908163265307|
|2014|    9|                153|            1746|  5.94388914656772|
|2014|   10|                181|            1983| 6.182357517034937|
|2014|   11|                157|  

## Moving Averages

This allows us to:
- Reduce daily volatility by smoothing out single day variations
- Reveal underlying trends by focusing on activity over time
- Filter out the noise in the data by averaging out the random fluctuations

In [32]:
from pyspark.sql.window import Window

# Define window specifications for moving averages
window_7 = Window.orderBy("date").rowsBetween(-6, 0)
window_30 = Window.orderBy("date").rowsBetween(-29, 0)

# Calculate moving averages
daily_with_ma = (daily_activity
    .withColumn("ma_7_submissions", F.avg("submission_count").over(window_7))
    .withColumn("ma_30_submissions", F.avg("submission_count").over(window_30))
    .withColumn("ma_7_comments", F.avg("total_comments").over(window_7))
    .withColumn("ma_30_comments", F.avg("total_comments").over(window_30))
)

daily_with_ma.select("date", "submission_count", "total_comments", "ma_7_submissions", "ma_7_comments").show(20)

25/11/20 23:52:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 155:======================================>                  (4 + 2) / 6]

+----------+----------------+--------------+-----------------+------------------+
|      date|submission_count|total_comments| ma_7_submissions|     ma_7_comments|
+----------+----------------+--------------+-----------------+------------------+
|2013-12-31|               4|            89|              4.0|              89.0|
|2014-01-01|               6|            56|              5.0|              72.5|
|2014-01-02|              10|           134|6.666666666666667|              93.0|
|2014-01-03|               3|            71|             5.75|              87.5|
|2014-01-04|               1|             3|              4.8|              70.6|
|2014-01-05|               3|             8|              4.5|60.166666666666664|
|2014-01-06|               8|            37|              5.0|56.857142857142854|
|2014-01-07|               4|            69|              5.0|              54.0|
|2014-01-08|               6|            33|              5.0|50.714285714285715|
|2014-01-09|    

25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:52:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Exponential Moving Averages

Consider the moving averages but with more weight given to the recent observations. With alpha 0.3, 30% of the weight goes to current day's value, while 70% goes to previous EMA (exponential moving average) value. This gives more influence to the recent data.

In [ ]:
# Exponential smoothing with alpha = 0.3
alpha = 0.3

# Collect data to compute exponentially smoothed values
daily_pandas = daily_activity.select("date", "submission_count", "total_comments").orderBy("date").toPandas()

# Apply exponential smoothing
daily_pandas['ema_submissions'] = daily_pandas['submission_count'].ewm(alpha=alpha, adjust=False).mean()
daily_pandas['ema_comments'] = daily_pandas['total_comments'].ewm(alpha=alpha, adjust=False).mean()

# Convert back to Spark DataFrame
daily_with_ema = spark.createDataFrame(daily_pandas)
daily_with_ema.show(20)

[Stage 175:>                                                        (0 + 1) / 1]

+----------+----------------+--------------+------------------+------------------+
|      date|submission_count|total_comments|   ema_submissions|      ema_comments|
+----------+----------------+--------------+------------------+------------------+
|2013-12-31|               4|            89|               4.0|              89.0|
|2014-01-01|               6|            56|               4.6|              79.1|
|2014-01-02|              10|           134|              6.22|             95.57|
|2014-01-03|               3|            71|             5.254| 88.19899999999998|
|2014-01-04|               1|             3|3.9777999999999993|62.639299999999984|
|2014-01-05|               3|             8| 3.684459999999999|46.247509999999984|
|2014-01-06|               8|            37| 4.979121999999999| 43.47325699999999|
|2014-01-07|               4|            69|4.6853853999999995| 51.13127989999999|
|2014-01-08|               6|            33|5.0797697799999995| 45.69189592999999|
|201

## Trend Detection and Decomposition

Analyse the long-term patters in the data

### Linear Trend Detection

Use a linear regression model to identify the overall trend

In [35]:
# Linear trend using window functions for submissions
window_spec = Window.orderBy("date")

trend_data = (daily_activity
    .withColumn("day_index", F.row_number().over(window_spec))
    .withColumn("submission_count_double", F.col("submission_count").cast("double"))
)

# Calculate linear trend coefficients
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(inputCols=["day_index"], outputCol="features")
trend_features = assembler.transform(trend_data)

lr = LinearRegression(featuresCol="features", labelCol="submission_count_double")
lr_model = lr.fit(trend_features)

print(f"Submission Trend coefficient (slope): {lr_model.coefficients[0]:.4f}")
print(f"Intercept: {lr_model.intercept:.4f}")

# Add trend predictions
trend_predictions = lr_model.transform(trend_features)
trend_predictions.select("date", "submission_count", "total_comments", "prediction").show(20)

25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 2

Submission Trend coefficient (slope): 0.1099
Intercept: -15.7653


25/11/20 23:53:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

+----------+----------------+--------------+-------------------+
|      date|submission_count|total_comments|         prediction|
+----------+----------------+--------------+-------------------+
|2013-12-31|               4|            89|-15.655430993134766|
|2014-01-01|               6|            56| -15.54554513351444|
|2014-01-02|              10|           134| -15.43565927389411|
|2014-01-03|               3|            71|-15.325773414273783|
|2014-01-04|               1|             3|-15.215887554653454|
|2014-01-05|               3|             8|-15.106001695033125|
|2014-01-06|               8|            37|-14.996115835412798|
|2014-01-07|               4|            69| -14.88622997579247|
|2014-01-08|               6|            33|-14.776344116172142|
|2014-01-09|              12|           116|-14.666458256551813|
|2014-01-10|               5|            47|-14.556572396931486|
|2014-01-11|               2|            69|-14.446686537311157|
|2014-01-12|             

25/11/20 23:53:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:53:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


The positive slope (0.1099) indicates that the subreddit is growing at around ~0.11 submissions per day, which would also translate to around 0.11 * 365 ≈ 40 more submissions annually

The negative intercept (-15.7653) predicts negative submissions on day 0 (December 31st, 2013). This indicates that the model underpredicts the activity in the early days, which can be explained by the high variance in the data (ranging from 1-12 submissions).

A linear model might be too simplistic for the data. One potential solution could be to use a polynomial or exponential model to better capture the complex nature of the data.

### Seasonality Detection

Detecting recurring patterns on different time scales.

In [36]:
# Detect seasonality by day of week
daily_with_dow = (daily_activity
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("month", F.month("date"))
)

# Average activity by day of week
dow_patterns = (daily_with_dow
    .groupBy("day_of_week")
    .agg(
        F.avg("submission_count").alias("avg_submissions"),
        F.avg("total_comments").alias("avg_comments")
    )
    .orderBy("day_of_week")
)

dow_patterns.show()

# Monthly seasonal patterns
monthly_patterns = (daily_with_dow
    .groupBy("month")
    .agg(
        F.avg("submission_count").alias("avg_submissions"),
        F.avg("total_comments").alias("avg_comments")
    )
    .orderBy("month")
)

monthly_patterns.show()

+-----------+------------------+------------------+
|day_of_week|   avg_submissions|      avg_comments|
+-----------+------------------+------------------+
|          1|10.184466019417476| 183.3495145631068|
|          2|27.119266055045873|431.70642201834863|
|          3|30.663636363636364| 501.2909090909091|
|          4| 33.46788990825688| 526.7981651376147|
|          5|32.872727272727275|505.77272727272725|
|          6|29.064814814814813|451.25925925925924|
|          7|11.653061224489797|216.87755102040816|
+-----------+------------------+------------------+



[Stage 226:============================>                            (3 + 3) / 6]

+-----+------------------+------------------+
|month|   avg_submissions|      avg_comments|
+-----+------------------+------------------+
|    1| 8.466666666666667|111.96666666666667|
|    2|11.196428571428571|113.82142857142857|
|    3|11.238095238095237|116.11111111111111|
|    4| 45.94252873563219| 972.5977011494252|
|    5| 55.41304347826087|1046.8369565217392|
|    6|15.716666666666667|162.43333333333334|
|    7|23.610169491525422|272.50847457627117|
|    8|26.610169491525422| 306.7118644067797|
|    9|20.724137931034484| 289.1896551724138|
|   10| 22.35483870967742|306.96774193548384|
|   11|22.694915254237287| 365.1186440677966|
|   12|              4.75|           47.8125|
+-----+------------------+------------------+



Weekly Patterns:
- Tuesday to Thursday (3-5) have the highest activity in the week, and Monday and Friday (2 and 6) also show fairly strong activity
- Weekends (1 and 7) show low activity, which is expected due to closure of markets.


Monthly Patterns:
- There's a strong surge in the spring (April and May), which could be attributed to various factors like the end of tax season and Q1 earnings reports
- June shows a sharp decline, a classic phenomenon in trading known as the summer slowdown, which then slowly recovers over July and August
- There's moderately stable activity in the Fall, showcasing normal trading patterns
- Winters show the lowest activity, with December having the lowest numbers due to the holiday season. Q1 follows the low activity and acts as the recovery period, while also preparing for the transition to the Q1 financial reports coming out in the Spring.

## Autoregression and Lag Features

Autoregression allows us to use past activity patterns to help predict the future activity. Values from previous time periods, called lag features, are used. The use of these lag features reveal how the momentum gets carried through over short or long term trends.

In [37]:
# Create lag features for submissions
window_lag = Window.orderBy("date")

lagged_data = (daily_activity
    .withColumn("lag_1", F.lag("submission_count", 1).over(window_lag))
    .withColumn("lag_7", F.lag("submission_count", 7).over(window_lag))
    .withColumn("lag_30", F.lag("submission_count", 30).over(window_lag))
    .dropna()
)

lagged_data.select("date", "submission_count", "lag_1", "lag_7", "lag_30").show(20)

25/11/20 23:54:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 232:======================================>                  (4 + 2) / 6]

+----------+----------------+-----+-----+------+
|      date|submission_count|lag_1|lag_7|lag_30|
+----------+----------------+-----+-----+------+
|2014-02-01|               5|    6|    6|     4|
|2014-02-02|               4|    5|    6|     6|
|2014-02-03|               7|    4|   20|    10|
|2014-02-04|              13|    7|    6|     3|
|2014-02-05|              14|   13|   14|     1|
|2014-02-06|              23|   14|   24|     3|
|2014-02-07|              10|   23|    6|     8|
|2014-02-08|               7|   10|    5|     4|
|2014-02-09|               3|    7|    4|     6|
|2014-02-10|              18|    3|    7|    12|
|2014-02-11|              18|   18|   13|     5|
|2014-02-12|               9|   18|   14|     2|
|2014-02-13|              18|    9|   23|     1|
|2014-02-14|              12|   18|   10|     2|
|2014-02-15|               2|   12|    7|     8|
|2014-02-16|               4|    2|    3|     7|
|2014-02-17|               9|    4|   18|    11|
|2014-02-18|        

25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [38]:
# Fit autoregressive model for submissions
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler_ar = VectorAssembler(
    inputCols=["lag_1", "lag_7", "lag_30"], 
    outputCol="features"
)

ar_features = assembler_ar.transform(lagged_data)

lr_ar = LinearRegression(featuresCol="features", labelCol="submission_count")
ar_model = lr_ar.fit(ar_features)

print(f"AR Model Coefficients: {ar_model.coefficients}")
print(f"AR Model RMSE: {ar_model.summary.rootMeanSquaredError:.2f}")
print(f"AR Model R2: {ar_model.summary.r2:.4f}")

25/11/20 23:54:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 2

AR Model Coefficients: [0.5136191264216219,0.5492924445003642,-0.1459906723671324]
AR Model RMSE: 15.66
AR Model R2: 0.8284


25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


The coefficients for short-term (yesterday), mid-term (weekly), and long-term (monthly) lag features are 0.514, 0.549 and -0.146 respectively.

This indicates that the short to mid term features have the strongest positive influence on predicting the current day's activity. Whereas, the long term features have a negative coefficient, which indicates a mean reversion effect. Thus, high activity in the 30 day lag feature translates to low activity in the current day, and vice versa, which captures the fluctuating nature of the activity around financial discourse

RMSE of 15.66 and R2 of 82.84% also indicate a strong fit of the model on the data.

## Change Point Detection

Detecting time periods with anomalous activity using z-scores. These could be due to various viral events like market crashes or meme stock movements, political or financial surprises in major firms, or collective movements.

In [39]:
# Calculate rolling statistics for change detection (using submissions)
window_60 = Window.orderBy("date").rowsBetween(-59, 0)

change_detection = (daily_activity
    .withColumn("rolling_mean", F.avg("submission_count").over(window_60))
    .withColumn("rolling_std", F.stddev("submission_count").over(window_60))
    .withColumn("z_score", 
                (F.col("submission_count") - F.col("rolling_mean")) / F.col("rolling_std"))
)

# Identify anomalies (z-score > 3)
anomalies = change_detection.filter(F.abs(F.col("z_score")) > 3)
print(f"Number of detected anomalies: {anomalies.count()}")
anomalies.select("date", "submission_count", "total_comments", "rolling_mean", "z_score").show(20)

25/11/20 23:54:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 2

Number of detected anomalies: 14


25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 288:======================================>                  (4 + 2) / 6]

+----------+----------------+--------------+------------------+------------------+
|      date|submission_count|total_comments|      rolling_mean|           z_score|
+----------+----------------+--------------+------------------+------------------+
|2015-01-14|              14|           138| 4.916666666666667|3.2300306766798155|
|2015-01-21|              15|           212|               5.1|3.1216731533003905|
|2015-01-23|              16|           162|               5.4| 3.063419875948013|
|2015-01-26|              18|           231|              5.75|3.2137223094019944|
|2015-01-27|              34|           558| 6.266666666666667| 5.273555058988071|
|2015-01-29|              27|           254| 6.916666666666667|3.3722551093572948|
|2015-03-11|              39|           392|12.366666666666667| 3.563381501175225|
|2015-08-21|             109|           717|              40.5|3.1683074598240606|
|2015-08-24|             117|          1054|41.666666666666664|3.1598166013999145|
|201

25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:54:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 2

## Stock Ticker Trends Over Time

Trends for popular stock tickers like meme stocks (GME and AMC), tech giants (like MSFT and NVDA), and market indices (like SPY and QQQ)

In [40]:
# Extract ticker mentions from submission titles and text
from pyspark.sql import types as T

tickers = ["AAPL", "TSLA", "NVDA", "GME", "AMC", "MSFT", "SPY", "QQQ"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_tickers(text):
    if not text:
        return []
    text_upper = text.upper()
    found = [t for t in ticker_set if t in text_upper]
    return found

# Apply to submissions (title + selftext)
submissions_with_tickers = (submissionsDF_clean
    .withColumn("date", F.to_date("created_ts"))
    .withColumn("combined_text", F.concat_ws(" ", F.col("title"), F.coalesce(F.col("selftext"), F.lit(""))))
    .withColumn("tickers", extract_tickers(F.col("combined_text")))
    .filter(F.size("tickers") > 0)
    .withColumn("ticker", F.explode("tickers"))
)

# Count ticker mentions by day
ticker_time_series = (submissions_with_tickers
    .groupBy("date", "ticker")
    .agg(F.count("*").alias("mention_count"))
    .orderBy("date", "ticker")
)

ticker_time_series.show(30)

[Stage 300:===============================================>         (5 + 1) / 6]

+----------+------+-------------+
|      date|ticker|mention_count|
+----------+------+-------------+
|2014-01-02|   SPY|            1|
|2014-01-05|  AAPL|            1|
|2014-01-09|  AAPL|            1|
|2014-01-16|  TSLA|            1|
|2014-01-23|  AAPL|            1|
|2014-01-24|  AAPL|            1|
|2014-01-27|  AAPL|            2|
|2014-01-28|  AAPL|            1|
|2014-01-28|   SPY|            1|
|2014-01-29|   SPY|            1|
|2014-01-30|  AAPL|            1|
|2014-01-30|   SPY|            1|
|2014-01-30|  TSLA|            1|
|2014-01-31|  AAPL|            1|
|2014-02-03|   SPY|            1|
|2014-02-03|  TSLA|            1|
|2014-02-07|  AAPL|            1|
|2014-02-08|  MSFT|            1|
|2014-02-08|   SPY|            1|
|2014-02-10|  AAPL|            1|
|2014-02-11|  AAPL|            2|
|2014-02-11|  TSLA|            2|
|2014-02-13|  AAPL|            1|
|2014-02-13|  TSLA|            2|
|2014-02-16|  AAPL|            1|
|2014-02-17|  TSLA|            1|
|2014-02-19|  

In [41]:
# Moving average for individual tickers
ticker_window = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)

ticker_smoothed = (ticker_time_series
    .withColumn("ma_7", F.avg("mention_count").over(ticker_window))
)

# Show trends for GME
ticker_smoothed.filter(F.col("ticker") == "GME").show(30)

+----------+------+-------------+------------------+
|      date|ticker|mention_count|              ma_7|
+----------+------+-------------+------------------+
|2014-08-18|   GME|            2|               2.0|
|2014-10-29|   GME|            1|               1.5|
|2014-11-21|   GME|            1|1.3333333333333333|
|2014-12-10|   GME|            1|              1.25|
|2015-01-14|   GME|            1|               1.2|
|2015-02-02|   GME|            1|1.1666666666666667|
|2015-03-13|   GME|            1|1.1428571428571428|
|2015-04-10|   GME|            1|               1.0|
|2015-05-05|   GME|            1|               1.0|
|2015-05-17|   GME|            1|               1.0|
|2015-05-28|   GME|            1|               1.0|
|2015-05-29|   GME|            1|               1.0|
|2015-05-31|   GME|            1|               1.0|
|2015-08-12|   GME|            1|               1.0|
|2015-08-13|   GME|            1|               1.0|
|2015-08-25|   GME|            1|             

## Cross-Correlation Analysis

Analysis of the relationship between different activity metrics like number of submissions, number of comments, average score of the submissions, etc

In [42]:
# Compute correlation between submissions and total comments
correlation_data = daily_activity.select("submission_count", "total_comments", "avg_score").toPandas()
correlation = correlation_data.corr()
print("Correlation Matrix:")
print(correlation)

# Lagged correlation - does submission count predict future comments?
lagged_corr_data = (daily_activity
    .withColumn("submissions_lag_1", F.lag("submission_count", 1).over(window_lag))
    .withColumn("comments_lag_1", F.lag("total_comments", 1).over(window_lag))
    .dropna()
    .select("submission_count", "total_comments", "submissions_lag_1", "comments_lag_1")
    .toPandas()
)

print("\nCorrelation with lagged values:")
print(lagged_corr_data.corr())

Correlation Matrix:
                  submission_count  total_comments  avg_score
submission_count          1.000000        0.950221   0.573700
total_comments            0.950221        1.000000   0.684494
avg_score                 0.573700        0.684494   1.000000


25/11/20 23:55:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 323:======================================>                  (4 + 2) / 6]


Correlation with lagged values:
                   submission_count  total_comments  submissions_lag_1  \
submission_count           1.000000        0.950222           0.850760   
total_comments             0.950222        1.000000           0.835113   
submissions_lag_1          0.850760        0.835113           1.000000   
comments_lag_1             0.843439        0.865046           0.950311   

                   comments_lag_1  
submission_count         0.843439  
total_comments           0.865046  
submissions_lag_1        0.950311  
comments_lag_1           1.000000  


25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/20 23:55:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Same Day Correlations:
- There's a high positive correlation between submissions count and total comments, which is as expected, as more posts lead to more discussions.
- There's a strong correlation between total number of comments and the highest average score for a post too, which indicates that the quality of the posts drive engagement, both in terms of upvotes and discussions.
- There's a moderate correlation between the number of posts and the average score of each post. This is slightly surprising, as it indicates that days with high volume of posts do not lead to the diluted quality of these posts.

Lagged Correlations:
- There is a strong correlation between the current day number of submissions and the day 1 lag number of submissions, which confirms the Autoregression model's predictions that activity carries over momentum over short periods of time.
- There is also strong correlation between number of comments and number of submissions with day 1 lag, implying that the posts need time to accumulate comments and discussions.
- Strong correlation between day 1 lag's number of comments and current day's number of submissions also hints towards an existence of a feedback loop - that active discussions lead to new posts in the short term

## Summary Statistics and Insights

Understanding the trajectory of growth of the subreddit's activity over the considered time period.

In [44]:
# Overall time series statistics
print("=== Time Series Summary ===")
print(f"Total days analyzed: {daily_activity.count()}")
print(f"Date range: {daily_activity.agg(F.min('date')).collect()[0][0]} to {daily_activity.agg(F.max('date')).collect()[0][0]}")

# Activity statistics
activity_stats = daily_activity.select(
    F.mean("submission_count").alias("avg_daily_submissions"),
    F.stddev("submission_count").alias("std_daily_submissions"),
    F.max("submission_count").alias("max_daily_submissions"),
    F.mean("total_comments").alias("avg_daily_comments"),
    F.stddev("total_comments").alias("std_daily_comments"),
    F.max("total_comments").alias("max_daily_comments"),
    F.mean("avg_comments_per_post").alias("avg_comments_per_post")
).collect()[0]

print(f"\nAverage daily submissions: {activity_stats['avg_daily_submissions']:.2f}")
print(f"Std dev daily submissions: {activity_stats['std_daily_submissions']:.2f}")
print(f"Max daily submissions: {activity_stats['max_daily_submissions']}")

print(f"\nAverage daily comments (total): {activity_stats['avg_daily_comments']:.2f}")
print(f"Average comments per post: {activity_stats['avg_comments_per_post']:.2f}")

print(f"Std dev daily comments: {activity_stats['std_daily_comments']:.2f}")
print(f"Max daily comments: {activity_stats['max_daily_comments']}")

=== Time Series Summary ===


Total days analyzed: 747


Date range: 2013-12-31 to 2018-06-01


[Stage 353:===================>                                     (2 + 4) / 6]


Average daily submissions: 25.33
Std dev daily submissions: 37.25
Max daily submissions: 211

Average daily comments (total): 407.13
Average comments per post: 12.64
Std dev daily comments: 779.03
Max daily comments: 5067


The standard deviation is higher than the mean for submissions, indicating towards a highly volatile community in terms of activity. Due to the event-driven nature of the financial discourse, this is aligned with the expected behavior.

In [45]:
# Year-over-year growth
yearly_stats = (daily_activity
    .withColumn("year", F.year("date"))
    .groupBy("year")
    .agg(
        F.sum("submission_count").alias("total_submissions"),
        F.sum("total_comments").alias("total_comments")
    )
    .orderBy("year")
)

print("\n=== Year-over-Year Activity ===")
yearly_stats.show()

# Calculate growth rates
yearly_pandas = yearly_stats.toPandas()
yearly_pandas['submission_growth'] = yearly_pandas['total_submissions'].pct_change() * 100
yearly_pandas['comment_growth'] = yearly_pandas['total_comments'].pct_change() * 100

print("\nYear-over-Year Growth Rates (%):")
print(yearly_pandas[['year', 'submission_growth', 'comment_growth']])


=== Year-over-Year Activity ===


+----+-----------------+--------------+
|year|total_submissions|total_comments|
+----+-----------------+--------------+
|2013|                4|            89|
|2014|             2133|         21613|
|2015|             9211|        117894|
|2018|             7575|        164533|
+----+-----------------+--------------+



[Stage 371:===============================================>         (5 + 1) / 6]


Year-over-Year Growth Rates (%):
   year  submission_growth  comment_growth
0  2013                NaN             NaN
1  2014       53225.000000    24184.269663
2  2015         331.833099      445.477259
3  2018         -17.761372       39.560113


The growth of total comments per submission across the years indicate that the users, over time, are having deeper discussions surrounding topics. This data is also from the pre-mainstream era of the subreddit (prior to the GME short squeeze days), and thus shows the maturity and popularity of the subreddit.

The analysis on growth hints towards a quick increase in the subreddit's activity going from 2014 to 2015. The anomalous growth rates and activity can be explained by the fact that 2014 was the start year in the data (with one additional day in 2013), and there is a large chunk of missing data from 2016-2018, which would further help analyse long term patterns.